In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install --upgrade pip

In [ ]:
!pip install pyarrow==16.1.0
!pip install datasets==2.20.0

In [ ]:
!pip install transformers==4.41.2
!pip install evaluate==0.4.1

In [ ]:
!pip install nltk==3.8.1

In [ ]:
!pip install rouge_score

In [ ]:
!pip install transformers==4.46.3
!pip install peft==0.11.1

In [ ]:
!pip uninstall -y transformers peft datasets pyarrow
!pip uninstall -y transformers peft datasets pyarrow

In [ ]:
!pip install pyarrow==16.1.0
!pip install datasets==2.20.0
!pip install transformers==4.36.2
!pip install peft==0.8.2
!pip install evaluate==0.4.1
!pip install nltk==3.8.1
!pip install rouge_score


In [ ]:
!pip install -U accelerate==0.34.2 transformers==4.44.2

In [ ]:
!pip install --upgrade accelerate==0.33.0 transformers==4.41.2

In [2]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()



In [ ]:
!nvidia-smi

In [3]:
import os, json, math, random, time
import numpy as np
import torch
import transformers
from datasets import load_dataset
import re
from transformers import pipeline
import nltk
nltk.download("punkt")
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize
from transformers import BartTokenizer
from datasets import load_from_disk
from transformers import (BartForConditionalGeneration, Trainer,
                          TrainingArguments, BartTokenizer,
                          Seq2SeqTrainer, DataCollatorForSeq2Seq,
                          Seq2SeqTrainingArguments
                          )
import evaluate

torch.cuda.empty_cache()

2025-11-20 00:34:10.538137: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763598850.560931    1541 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763598850.567894    1541 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
raw_dataset = load_dataset("ccdv/arxiv-summarization")
train_data = raw_dataset["train"].train_test_split(test_size=0.8, shuffle=True)["train"]
val_data = raw_dataset["validation"].train_test_split(test_size=0.8, shuffle=True)["train"]

print("Train size:", len(train_data))
print("Validation size:", len(val_data))

In [ ]:
def extract_section(text, section_name):
    pattern = rf"\\section\{{.*?{section_name}.*?\}}(.*?)(?=\\section|\Z)"
    match = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
    return match.group(1).strip() if match else ""

In [ ]:
def extract_equations(text, max_eq=6):
    return re.findall(
        r'\$.*?\$|\\begin\{equation\}.*?\\end\{equation\}',
        text,
        re.DOTALL
    )[:max_eq]

In [ ]:
def explain_equations(text, equations, window=2):
    sentences = sent_tokenize(text)

    explanations = []

    for eq in equations[:5]:
        eq_short = eq[:30]
        expl = "No explanation found."

        for i, s in enumerate(sentences):
            if eq_short in s:
                start = max(0, i - window)
                end = min(len(sentences), i + window + 1)
                expl = " ".join(sentences[start:end])
                break

        explanations.append(expl)

    return explanations

In [ ]:
method_keywords = [
    "methodology", "methods", "approach", "proposed method",
    "model", "technique", "experimental setup"
]

def extract_methodology(text, max_sentences=6):
    sentences = sent_tokenize(text)

    method_sents = []

    science_verbs = [
        r"we propose", r"we develop", r"we introduce",
        r"our model", r"our approach", r"this method",
        r"this model", r"our technique", r"we design"
    ]

    # Extract sentences containing section-like keywords
    for s in sentences:
        if any(re.search(rf"(?i){kw}", s) for kw in method_keywords):
            method_sents.append(s)

    # Extract methodological sentences if section not found
    if len(method_sents) < 3:
        for s in sentences:
            if any(re.search(v, s, re.IGNORECASE) for v in science_verbs):
                method_sents.append(s)

    blacklist = ["related work", "introduction", "references", "\\section"]
    method_sents = [
        s for s in method_sents
        if not any(bl in s.lower() for bl in blacklist)
    ]

    if not method_sents:
        return "No clear methodology — inferred approach from context"

    return " ".join(method_sents[:max_sentences])

In [ ]:
def extract_results(text):
    patterns = [r"(?i)result", r"(?i)experiment", r"(?i)evaluation"]
    sentences = sent_tokenize(text)

    result_lines = []
    for s in sentences:
        if any(re.search(p, s) for p in patterns):
            result_lines.append(s)

    if not result_lines:
        return "No clear results"

    return " ".join(result_lines[:5])

In [ ]:
def build_structured(batch):
    article = batch["article"]

    equations = extract_equations(article)
    eq_explanations = explain_equations(article, equations)

    return {
        "text": article,
        "core": batch["abstract"],
        "method": extract_methodology(article),
        "equations": " | ".join(extract_equations(article)),
        "equation_explanation": " | ".join(eq_explanations),
        "results": extract_results(article)
    }

In [ ]:
processed_train = train_data.map(build_structured)
processed_val = val_data.map(build_structured)

In [4]:
tokenizer = BartTokenizer.from_pretrained("facebook/bart-large")

In [5]:
def preprocess(batch):
    source = (
        "Summarize into structured sections:\n\n"
        "Core Idea:\nMethodology:\nKey Equation(s):\nEquation Explanation:\nResults:\n\n"
        "Text:\n" + batch["text"]
    )

    target = (
        f"Core Idea: {batch['core']}\n"
        f"Methodology: {batch['method']}\n"
        f"Key Equation(s): {batch['equations']}\n"
        f"Equation Explanation: {batch['equation_explanation']}\n"
        f"Results: {batch['results']}"
    )

    model_inputs = tokenizer(
        source,
        max_length=512,
        truncation=True
    )

    labels = tokenizer(
        target,
        max_length=256,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
train_dataset = processed_train.map(preprocess, batched=False, batch_size=64)
val_dataset = processed_val.map(preprocess, batched=False, batch_size=64)

In [ ]:
train_dataset.save_to_disk("/kaggle/working/train_dataset")
val_dataset.save_to_disk("/kaggle/working/val_dataset")

In [6]:
## load saved dataset
train_dataset = load_from_disk("/kaggle/working/train_dataset")
val_dataset = load_from_disk("/kaggle/working/val_dataset")

In [7]:
# model = BartForConditionalGeneration.from_pretrained("facebook/bart-large")
# model.config.use_cache = False
# model.gradient_checkpointing_enable()

model = BartForConditionalGeneration.from_pretrained("facebook/bart-base")
model.config.use_cache = False              
model.gradient_checkpointing_enable()

In [8]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

### Compute Matrics

In [9]:
rouge = evaluate.load("rouge")

def postprocess_text(preds, labels):
    preds = [p.strip() for p in preds]
    labels = [l.strip() for l in labels]
    return preds, labels

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    result = {k: float(v.mid.fmeasure * 100) for k, v in result.items()}
    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
    result["gen_len"] = float(np.mean(prediction_lens))
    return result

In [ ]:
model.gradient_checkpointing_enable()   # saves huge GPU memory
model.config.use_cache = False          
torch.cuda.empty_cache() 

In [10]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./bart_ckpt",
    max_steps=2000,                 # <-- KEY
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,  # effective batch 4
    eval_steps=500,
    logging_steps=200,
    learning_rate=3e-5,
    fp16=True,
    report_to="none",
    optim="adafactor",
)

In [11]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator
)

/usr/local/lib/python3.11/dist-packages/accelerate/accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
max_steps is given, it will override any value given in num_train_epochs


In [12]:
trainer.train()

/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss
200,3.477500
400,3.150500
600,3.065400
800,3.017400
1000,2.987900
1200,2.935800
1400,2.909000
1600,2.883900
1800,2.901900
2000,2.867900


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0, 'forced_eos_token_id': 2}
Your generation config was originally created from the model config, but the model config has changed since then. Unless you pass the `generation_config` argument to this model's `generate` calls, they will revert to the legacy behavior where the base `generate` parameterization is loaded from the model config instead. To avoid this behavior and this warning, we recommend you to overwrite the generation config model attribute before calling the model's `save_pretrained`, preferably also removing any generation kwargs from the model config. This warni

TrainOutput(global_step=2000, training_loss=3.019714416503906, metrics={'train_runtime': 15380.6859, 'train_samples_per_second': 1.04, 'train_steps_per_second': 0.13, 'total_flos': 4877570125209600.0, 'train_loss': 3.019714416503906, 'epoch': 0.39401103230890466})

TrainOutput(global_step=2000, training_loss=3.019714416503906, metrics={'train_runtime': 15380.6859, 'train_samples_per_second': 1.04, 'train_steps_per_second': 0.13, 'total_flos': 4877570125209600.0, 'train_loss': 3.019714416503906, 'epoch': 0.39401103230890466})

In [13]:
final_dir = "/kaggle/working//bart_model_final"
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)
print("Saved final model to", final_dir)

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0, 'forced_eos_token_id': 2}
Your generation config was originally created from the model config, but the model config has changed since then. Unless you pass the `generation_config` argument to this model's `generate` calls, they will revert to the legacy behavior where the base `generate` parameterization is loaded from the model config instead. To avoid this behavior and this warning, we recommend you to overwrite the generation config model attribute before calling the model's `save_pretrained`, preferably also removing any generation kwargs from the model config. This warni

Saved final model to /kaggle/working//bart_model_final


In [16]:
save_dir = "./bart_summarizes_final"

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0, 'forced_eos_token_id': 2}
Your generation config was originally created from the model config, but the model config has changed since then. Unless you pass the `generation_config` argument to this model's `generate` calls, they will revert to the legacy behavior where the base `generate` parameterization is loaded from the model config instead. To avoid this behavior and this warning, we recommend you to overwrite the generation config model attribute before calling the model's `save_pretrained`, preferably also removing any generation kwargs from the model config. This warni

('./bart_summarizes_final/tokenizer_config.json',
 './bart_summarizes_final/special_tokens_map.json',
 './bart_summarizes_final/vocab.json',
 './bart_summarizes_final/merges.txt',
 './bart_summarizes_final/added_tokens.json')

In [17]:
import shutil

shutil.make_archive("bart_summarizes_final", "zip", save_dir)

'/kaggle/working/bart_summarizes_final.zip'